In [34]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

diabetes = pd.read_csv('data/diabetes_data.csv')
diabetes.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome,Gender
0,6,98,58,33,190,34.0,0.430,43,0,Female
1,2,112,75,32,0,35.7,0.148,21,0,Female
2,2,108,64,0,0,30.8,0.158,21,0,Female
3,8,107,80,0,0,24.6,0.856,34,0,Female
4,7,136,90,0,0,29.9,0.210,50,0,Female


In [35]:
# Ищем дубликаты и удаляем их
dupl_columns = list(diabetes.columns)
diabetes = diabetes.drop_duplicates(subset=dupl_columns)
print(f'Результирующее число записей без дубликатов: {diabetes.shape[0]}')



Результирующее число записей без дубликатов: 768


In [37]:
# Удаляем все неинформативные признаки (повторения — 95%, уникальные значение — 95%)

low_info_columns = []
for col in diabetes.columns:
    top_freq = diabetes[col].value_counts(normalize=True).max()
    nunique_ratio = diabetes[col].nunique() / diabetes[col].count()
    if top_freq > 0.95:
        low_info_columns.append(col)
    if nunique_ratio > 0.95:
        low_info_columns.append(col)
diabetes = diabetes.drop(low_info_columns, axis=1)
print(low_info_columns)

['Gender']


In [ ]:
# Ищем пропуски в данных
cols_null_percent = diabetes.isnull().mean() * 100
cols_with_null = cols_null_percent[cols_null_percent>0].sort_values(ascending=False)
display(cols_with_null)

In [38]:
# Функция, которая возвращает NaN, если найдет ноль в столбце
def replace_nulls(x):
    if x == 0:
        return np.nan
    return x

# Применяем функцию к столбцам
diabetes['Glucose'] = diabetes['Glucose'].apply(replace_nulls)
diabetes['BloodPressure'] = diabetes['BloodPressure'].apply(replace_nulls)
diabetes['SkinThickness'] = diabetes['SkinThickness'].apply(replace_nulls)
diabetes['Insulin'] = diabetes['Insulin'].apply(replace_nulls)
diabetes['BMI'] = diabetes['BMI'].apply(replace_nulls)

# Выводим долю пропусков, округленную до сотых
diabetes.isnull().mean().round(2).sort_values(ascending=False)

Insulin                     0.49
SkinThickness               0.30
BloodPressure               0.05
Glucose                     0.01
BMI                         0.01
Pregnancies                 0.00
DiabetesPedigreeFunction    0.00
Age                         0.00
Outcome                     0.00
dtype: float64

In [41]:
# Удаляем признаки, где число пропусков составляет более 30%
thresh = diabetes.shape[0]*0.7
diabetes = diabetes.dropna(thresh=thresh, axis=1)

print(diabetes.shape[1])
print(diabetes)

8
     Pregnancies  Glucose  BloodPressure  SkinThickness   BMI  \
0              6     98.0           58.0           33.0  34.0   
1              2    112.0           75.0           32.0  35.7   
2              2    108.0           64.0            NaN  30.8   
3              8    107.0           80.0            NaN  24.6   
4              7    136.0           90.0            NaN  29.9   
..           ...      ...            ...            ...   ...   
763            5    139.0           64.0           35.0  28.6   
764            1     96.0          122.0            NaN  22.4   
765           10    101.0           86.0           37.0  45.6   
766            0    141.0            NaN            NaN  42.4   
767            0    125.0           96.0            NaN  22.5   

     DiabetesPedigreeFunction  Age  Outcome  
0                       0.430   43        0  
1                       0.148   21        0  
2                       0.158   21        0  
3                       0.856   3

In [40]:
# Удаляем строки, в которых содержится более двух пропусков одновременно
m = diabetes.shape[1]
diabetes = diabetes.dropna(thresh=m-2, axis=0)

print(f'Результирующее число записей: {diabetes.shape[0]}')

Результирующее число записей: 761


In [44]:
# В оставшихся записях заменяем пропуски на медиану
values = {
    'SkinThickness': diabetes['SkinThickness'].median(),
    'BloodPressure': diabetes['BloodPressure'].median(),
    'Glucose': diabetes['Glucose'].median(),
    'BMI': diabetes['BMI'].median(),
    'DiabetesPedigreeFunction': diabetes['DiabetesPedigreeFunction'].median(),
    'Age': diabetes['Age'].median(),
    'Outcome': diabetes['Outcome'].median()
}

diabetes = diabetes.fillna(values)

display(diabetes['SkinThickness'].mean().round(1))

29.1

In [46]:
# Ищем выбросы с помощью классического метода межквартильного размаха
def outliers_iqr(data, feature):
    x = data[feature]
    quartile_1, quartile_3 = x.quantile(0.25), x.quantile(0.75),
    iqr = quartile_3 - quartile_1
    lower_bound = quartile_1 - (iqr * 1.5)
    upper_bound = quartile_3 + (iqr * 1.5)
    outliers = data[(x < lower_bound) | (x > upper_bound)]
    cleaned = data[(x >= lower_bound) & (x <= upper_bound)]
    return outliers, cleaned

outliers, cleaned = outliers_iqr(diabetes, 'SkinThickness')
print(f'Число выбросов по методу Тьюки: {outliers.shape[0]}')

Число выбросов по методу Тьюки: 87


In [47]:
# Ищем выбросы с помощью классического метода z-отклонения
def outliers_z_score(data, feature, log_scale=False):
    if log_scale:
        x = np.log(data[feature]+1)
    else:
        x = data[feature]
    mu = x.mean()
    sigma = x.std()
    lower_bound = mu - 3 * sigma
    upper_bound = mu + 3 * sigma
    outliers = data[(x < lower_bound) | (x > upper_bound)]
    cleaned = data[(x >= lower_bound) & (x <= upper_bound)]
    return outliers, cleaned

outliers, cleaned = outliers_z_score(diabetes, 'SkinThickness')
print(f'Число выбросов по методу z-отклонения: {outliers.shape[0]}')

Число выбросов по методу z-отклонения: 4


In [48]:
# Ищем выбросы с помощью классического метода межквартильного размаха

def outliers_iqr(data, feature):
    x = data[feature]
    quartile_1, quartile_3 = x.quantile(0.25), x.quantile(0.75),
    iqr = quartile_3 - quartile_1
    lower_bound = quartile_1 - (iqr * 1.5)
    upper_bound = quartile_3 + (iqr * 1.5)
    outliers = data[(x < lower_bound) | (x > upper_bound)]
    cleaned = data[(x >= lower_bound) & (x <= upper_bound)]
    return outliers, cleaned

outliers_quart, cleaned_quart = outliers_iqr(diabetes, 'DiabetesPedigreeFunction')

# Далее ищем выбросы с помощью межквартильного размаха в логарифмическом масштабе

def outliers_iqr_mod(data, feature, left=1.5, right=1.5, log_scale=False):
    if log_scale:
        x = np.log(data[feature])
    else:
        x = data[feature]
    quartile_1, quartile_3 = x.quantile(0.25), x.quantile(0.75)
    iqr = quartile_3 - quartile_1
    lower_bound = quartile_1 - (iqr * left)
    upper_bound = quartile_3 + (iqr * right)

    outliers = data[(x < lower_bound) | (x > upper_bound)]
    cleaned = data[(x >= lower_bound) & (x <= upper_bound)]
    return outliers, cleaned

outliers_log, cleaned_log = outliers_iqr_mod(diabetes, 'DiabetesPedigreeFunction')

outliers_diff = outliers_quart-outliers_log

print(f'Разница числа выбросов между двумя методами: {outliers_diff.shape[0]}')

Разница числа выбросов между двумя методами: 29
